In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')

from config import SPE1_PICKLE_ROOT, CELL_IDS, DICT_CELL_TYPE
from spikeparam.patch.fit import Spike

plt.rcParams['font.family'] = 'Helvetica Neue'

SPE1_PKL = SPE1_PICKLE_ROOT
PVC6_PKL = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles'

In [ ]:
FORCE_RERUN = False   # set True to refit pvc-6 and overwrite R² caches

## pvc-6 R² — refit from all_data pickles and cache

In [ ]:
def get_pvc6_r2(all_data_pkl, cache_pkl, fs=200000, force=False):
    """Fit Spike on pvc-6 all_data and cache r2_exp / r2_ramp."""
    if not force and os.path.exists(cache_pkl):
        with open(cache_pkl, 'rb') as f:
            d = pickle.load(f)
        # backfill old cache that used 'r_squared_exp' key
        if 'r_squared_exp' in d:
            d = {'r2_exp': d['r_squared_exp'], 'r2_ramp': d['r_squared_ramp']}
        return d

    with open(all_data_pkl, 'rb') as f:
        all_data = pickle.load(f)
    all_data_flat = [s for sweep in all_data for s in sweep]

    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=.008,
               pre_inflection_ms=0.5)
    sp.fit(all_data_flat, fs, n_jobs=-1, progress=tqdm)
    sp.gen_fit(ramp=True, exp=True)

    result = {
        'r2_exp':  np.asarray(sp.r_squared_exp,  dtype=float),
        'r2_ramp': np.asarray(sp.r_squared_ramp, dtype=float),
    }
    with open(cache_pkl, 'wb') as f:
        pickle.dump(result, f)
    print(f'Cached to {os.path.basename(cache_pkl)}')
    return result


pvc6_r2 = {
    'c1': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data.pkl'),
        os.path.join(PVC6_PKL, '_r2_c1.pkl'),
        force=FORCE_RERUN,
    ),
    'c2': get_pvc6_r2(
        os.path.join(PVC6_PKL, 'all_data2.pkl'),
        os.path.join(PVC6_PKL, '_r2_c2.pkl'),
        force=FORCE_RERUN,
    ),
}

for cid, d in pvc6_r2.items():
    print(f'pvc-6 {cid}: n={len(d["r2_exp"])}  '
          f'median r²_exp={np.nanmedian(d["r2_exp"]):.3f}  '
          f'median r²_ramp={np.nanmedian(d["r2_ramp"]):.3f}')

## spe-1 R² — load from spike_fit_pickles

In [ ]:
_fit_dir     = os.path.join(SPE1_PKL, 'spike_fit_pickles')
_cluster_dir = os.path.join(SPE1_PKL, 'cluster_pickles')

spe1_r2 = []   # list of dicts: cell_id, cell_type, r2_exp, r2_ramp

for _cid in CELL_IDS:
    # only include cells present in the main analysis (cluster pickles)
    _cluster_p = os.path.join(_cluster_dir, f'{_cid}_cluster_df.pkl')
    if not os.path.exists(_cluster_p):
        continue

    _fit_p = os.path.join(_fit_dir, f'{_cid}_spike_fit.pkl')
    if not os.path.exists(_fit_p):
        continue

    with open(_fit_p, 'rb') as _f:
        _sp = pickle.load(_f)

    if _sp.r_squared_exp is None:
        _sp.gen_fit(ramp=True, exp=True)

    _cnum  = int(_cid.replace('c', ''))
    _ctype = DICT_CELL_TYPE.get(_cnum, 'PC')
    spe1_r2.append({
        'cell_id':   _cid,
        'cell_type': _ctype,
        'r2_exp':    np.asarray(_sp.r_squared_exp,  dtype=float),
        'r2_ramp':   np.asarray(_sp.r_squared_ramp, dtype=float),
    })

print(f'Loaded {len(spe1_r2)} spe-1 cells (cluster-pickle gated)')
for d in spe1_r2:
    print(f"  {d['cell_id']} ({d['cell_type']}): n={len(d['r2_exp'])}  "
          f"med r²_exp={np.nanmedian(d['r2_exp']):.3f}  "
          f"med r²_ramp={np.nanmedian(d['r2_ramp']):.3f}")

## R² distributions — summary + per-cell

In [ ]:
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import pandas as pd

matplotlib.rcParams['font.family']     = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Helvetica Neue', 'Helvetica', 'Arial']

_FS_AX  = 34
_FS_TK  = 28
_FS_SM  = 18
_FS_MED = 24
_LW_SP  = 2.5
_ALPHA  = 0.82

COL_PVC6    = '#888888'
COL_SPE1    = '#5A5A80'
COL_SPE1_PC = '#3D3D65'
COL_SPE1_IN = '#8888B0'

ALL_FEATS  = ['ramp_amp', 'inflection_amp', 'inflection_time',
              'peak_amp', 'peak_width', 'peak_sharpness',
              'exp_lambda', 'exp_const', 'log_isi']
RAMP_FEATS = ['ramp_amp', 'inflection_time', 'inflection_amp']
EXP_FEATS  = ['exp_lambda', 'exp_const']

_POSITIVE_ONLY = {'exp_lambda'}
_CV_CAP = 10.0   # CV > 10 = degenerate (near-zero median, near-zero lambda, etc.)


def _rob_cv(vals, positive_only=False):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if positive_only:
        vals = vals[vals > 0]
    if len(vals) < 2:
        return np.nan
    med = np.median(vals)
    s   = np.std(vals, ddof=1)
    if np.abs(med) < 1e-12 or np.abs(med) < 0.01 * s:
        return np.nan
    cv = s / np.abs(med)
    if cv > _CV_CAP:
        return np.nan
    return cv


def _pool_spe1(r2_key):
    arrs = [d[r2_key][~np.isnan(d[r2_key])] for d in spe1_r2]
    return np.concatenate(arrs) if arrs else np.array([])

def _pool_pvc6(r2_key):
    return np.concatenate([d[r2_key][~np.isnan(d[r2_key])] for d in pvc6_r2.values()])

def _style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp in ['bottom', 'left']:
        ax.spines[sp].set_linewidth(_LW_SP)
    ax.tick_params(axis='both', width=_LW_SP, labelsize=_FS_TK)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')

def _boxes(ax, data, pos, cols, widths=0.55):
    bp = ax.boxplot(data, positions=pos, widths=widths, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2.2),
                    whiskerprops=dict(linewidth=_LW_SP * 0.9),
                    capprops=dict(linewidth=_LW_SP * 0.9),
                    showfliers=False, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='white',
                                   markeredgecolor='white', markersize=9, zorder=5))
    for p, c in zip(bp['boxes'], cols):
        p.set_facecolor(c); p.set_alpha(_ALPHA); p.set_linewidth(0)
    for w, c in zip(bp['whiskers'], [c for c in cols for _ in (0, 1)]):
        w.set_color(c); w.set_linewidth(_LW_SP * 0.9)
    for cp, c in zip(bp['caps'], [c for c in cols for _ in (0, 1)]):
        cp.set_color(c); cp.set_linewidth(_LW_SP * 0.9)

def _r2_yax(ax):
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0', '0.25', '0.5', '0.75', '1'], fontweight='bold', fontsize=_FS_TK)

def _grp_label(ax, pos_list, label, transform):
    ax.text(np.mean(pos_list), -0.12, label, ha='center', va='top',
            transform=transform, fontsize=_FS_TK, fontweight='bold', clip_on=False)

def _med_annot(ax, data_list, positions):
    for pos, data in zip(positions, data_list):
        med = np.nanmedian(data)
        ax.text(pos + 0.40, med, f'{med:.2f}', va='center', ha='left',
                fontsize=_FS_MED, fontweight='bold', color='#333333', clip_on=False)


# ── per-cell variability ──────────────────────────────────────────────────────
_fit_dir = os.path.join(SPE1_PKL, 'spike_fit_pickles')
cv_rows = []
for d in spe1_r2:
    cid = d['cell_id']
    with open(os.path.join(_fit_dir, f'{cid}_spike_fit.pkl'), 'rb') as _f:
        _sp = pickle.load(_f)
    df_f = _sp.df_features
    row = {'cid': cid, 'ct': d['cell_type'],
           'r2_exp':  np.nanmedian(d['r2_exp']),
           'r2_ramp': np.nanmedian(d['r2_ramp'])}
    for feat in ALL_FEATS:
        if feat in df_f.columns:
            row[f'cv_{feat}'] = _rob_cv(df_f[feat].values,
                                         positive_only=(feat in _POSITIVE_ONLY))
        else:
            row[f'cv_{feat}'] = np.nan
    cv_rows.append(row)
df_cv = pd.DataFrame(cv_rows)
df_cv['ramp_var'] = df_cv[[f'cv_{f}' for f in RAMP_FEATS]].median(axis=1)
df_cv['exp_var']  = df_cv[[f'cv_{f}' for f in EXP_FEATS]].median(axis=1)

_pc_cells = [d for d in spe1_r2 if d['cell_type'] == 'PC']
_in_cells = [d for d in spe1_r2 if d['cell_type'] == 'IN']


# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(28, 20))
outer = gridspec.GridSpec(3, 1, figure=fig, hspace=0.55, height_ratios=[1, 1, 1])
gs0 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0], wspace=0.18, width_ratios=[1, 2.4])
gs1 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[1], wspace=0.18, width_ratios=[1, 2.4])
gs2 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[2], wspace=0.35, width_ratios=[1, 1])


# A: ramp R², spe-1 vs pvc-6 ─────────────────────────────────────────────────
ax_a = fig.add_subplot(gs0[0])
_ramp_spe1 = _pool_spe1('r2_ramp')
_ramp_pvc6 = _pool_pvc6('r2_ramp')
_boxes(ax_a, [_ramp_spe1, _ramp_pvc6], [0, 1], [COL_SPE1, COL_PVC6], widths=0.65)
_med_annot(ax_a, [_ramp_spe1, _ramp_pvc6], [0, 1])
_r2_yax(ax_a)
ax_a.set_xlim(-0.65, 1.65)
ax_a.set_xticks([0, 1])
ax_a.set_xticklabels(['spe-1', 'pvc-6'], fontsize=_FS_TK, fontweight='bold')
ax_a.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
ax_a.set_title('Ramp fit', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_a)


# B: ramp R² per cell — sorted small→large ───────────────────────────────────
ax_b = fig.add_subplot(gs0[1])
_pc_sort_r = sorted(_pc_cells, key=lambda d: np.nanmedian(d['r2_ramp']))
_in_sort_r = sorted(_in_cells, key=lambda d: np.nanmedian(d['r2_ramp']))
b_data, b_cols, b_pos = [], [], []
_x = 0
for d in _pc_sort_r:
    v = d['r2_ramp'][~np.isnan(d['r2_ramp'])]
    b_data.append(v); b_cols.append(COL_SPE1_PC); b_pos.append(_x); _x += 1
_sep_b = _x - 0.5; _x += 0.8
for d in _in_sort_r:
    v = d['r2_ramp'][~np.isnan(d['r2_ramp'])]
    b_data.append(v); b_cols.append(COL_SPE1_IN); b_pos.append(_x); _x += 1
_boxes(ax_b, b_data, b_pos, b_cols)
ax_b.axvline(_sep_b, color='#dddddd', lw=1.5, ls='--')
_r2_yax(ax_b)
ax_b.set_xticks([])
ax_b.set_xlim(-0.5, _x - 0.2)
_grp_label(ax_b, b_pos[:len(_pc_sort_r)], f'PC  (n={len(_pc_sort_r)})', ax_b.get_xaxis_transform())
_grp_label(ax_b, b_pos[len(_pc_sort_r):], f'IN  (n={len(_in_sort_r)})', ax_b.get_xaxis_transform())
ax_b.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
_style(ax_b)


# C: exp R², spe-1 vs pvc-6 ──────────────────────────────────────────────────
ax_c = fig.add_subplot(gs1[0])
_exp_spe1 = _pool_spe1('r2_exp')
_exp_pvc6 = _pool_pvc6('r2_exp')
_boxes(ax_c, [_exp_spe1, _exp_pvc6], [0, 1], [COL_SPE1, COL_PVC6], widths=0.65)
_med_annot(ax_c, [_exp_spe1, _exp_pvc6], [0, 1])
_r2_yax(ax_c)
ax_c.set_xlim(-0.65, 1.65)
ax_c.set_xticks([0, 1])
ax_c.set_xticklabels(['spe-1', 'pvc-6'], fontsize=_FS_TK, fontweight='bold')
ax_c.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
ax_c.set_title('Exp. decay fit', fontsize=_FS_AX, fontweight='bold', pad=10, loc='left')
_style(ax_c)


# D: exp R² per cell — sorted small→large ────────────────────────────────────
ax_d = fig.add_subplot(gs1[1])
_pc_sort_e = sorted(_pc_cells, key=lambda d: np.nanmedian(d['r2_exp']))
_in_sort_e = sorted(_in_cells, key=lambda d: np.nanmedian(d['r2_exp']))
d_data, d_cols, d_pos = [], [], []
_x = 0
for d in _pc_sort_e:
    v = d['r2_exp'][~np.isnan(d['r2_exp'])]
    d_data.append(v); d_cols.append(COL_SPE1_PC); d_pos.append(_x); _x += 1
_sep_d = _x - 0.5; _x += 0.8
for d in _in_sort_e:
    v = d['r2_exp'][~np.isnan(d['r2_exp'])]
    d_data.append(v); d_cols.append(COL_SPE1_IN); d_pos.append(_x); _x += 1
_boxes(ax_d, d_data, d_pos, d_cols)
ax_d.axvline(_sep_d, color='#dddddd', lw=1.5, ls='--')
_r2_yax(ax_d)
ax_d.set_xticks([])
ax_d.set_xlim(-0.5, _x - 0.2)
_grp_label(ax_d, d_pos[:len(_pc_sort_e)], f'PC  (n={len(_pc_sort_e)})', ax_d.get_xaxis_transform())
_grp_label(ax_d, d_pos[len(_pc_sort_e):], f'IN  (n={len(_in_sort_e)})', ax_d.get_xaxis_transform())
ax_d.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
_style(ax_d)


# E: ramp feature variability vs. ramp R² ────────────────────────────────────
ax_e = fig.add_subplot(gs2[0])
for ct, col in [('PC', COL_SPE1_PC), ('IN', COL_SPE1_IN)]:
    sub = df_cv[df_cv['ct'] == ct]
    ax_e.scatter(sub['ramp_var'], sub['r2_ramp'], color=col, s=90,
                 alpha=0.85, zorder=3, edgecolors='none')
_xv = df_cv['ramp_var'].values
_yv = df_cv['r2_ramp'].values
_m  = ~(np.isnan(_xv) | np.isnan(_yv))
_r, _p = pearsonr(_xv[_m], _yv[_m])
_slope, _intercept = np.polyfit(_xv[_m], _yv[_m], 1)
_xx = np.linspace(_xv[_m].min(), _xv[_m].max(), 100)
ax_e.plot(_xx, _slope * _xx + _intercept, color='#999999', ls='--', lw=1.8, zorder=1, alpha=0.7)
_p_str = 'p < 0.001' if _p < 0.001 else f'p = {_p:.3f}'
ax_e.text(0.97, 0.05, f'r = {_r:.2f},  {_p_str}',
          transform=ax_e.transAxes, ha='right', va='bottom',
          fontsize=_FS_TK, fontweight='bold', color='#444444')
ax_e.set_xlabel('Ramp feature variability', fontsize=_FS_AX, fontweight='bold')
ax_e.set_ylabel('Median ramp R²', fontsize=_FS_AX, fontweight='bold')
_style(ax_e)
for lbl in ax_e.get_xticklabels() + ax_e.get_yticklabels(): lbl.set_fontweight('bold')


# F: exp feature variability vs. exp R² ──────────────────────────────────────
ax_f = fig.add_subplot(gs2[1])
for ct, col in [('PC', COL_SPE1_PC), ('IN', COL_SPE1_IN)]:
    sub = df_cv[df_cv['ct'] == ct]
    ax_f.scatter(sub['exp_var'], sub['r2_exp'], color=col, s=90,
                 alpha=0.85, zorder=3, edgecolors='none')
_xv2 = df_cv['exp_var'].values
_yv2 = df_cv['r2_exp'].values
_m2  = ~(np.isnan(_xv2) | np.isnan(_yv2))
_r2f, _p2 = pearsonr(_xv2[_m2], _yv2[_m2])
_slope2, _intercept2 = np.polyfit(_xv2[_m2], _yv2[_m2], 1)
_xx2 = np.linspace(_xv2[_m2].min(), _xv2[_m2].max(), 100)
ax_f.plot(_xx2, _slope2 * _xx2 + _intercept2, color='#999999', ls='--', lw=1.8, zorder=1, alpha=0.7)
_p_str2 = 'p < 0.001' if _p2 < 0.001 else f'p = {_p2:.3f}'
ax_f.text(0.97, 0.05, f'r = {_r2f:.2f},  {_p_str2}',
          transform=ax_f.transAxes, ha='right', va='bottom',
          fontsize=_FS_TK, fontweight='bold', color='#444444')
ax_f.set_xlabel('Exp. decay feature variability', fontsize=_FS_AX, fontweight='bold')
ax_f.set_ylabel('Median exp. R²', fontsize=_FS_AX, fontweight='bold')
_style(ax_f)
for lbl in ax_f.get_xticklabels() + ax_f.get_yticklabels(): lbl.set_fontweight('bold')

plt.show()

In [ ]:
# ── standalone legend ─────────────────────────────────────────────────────────
fig_leg, ax_leg = plt.subplots(figsize=(5, 1.6))
ax_leg.axis('off')
ax_leg.legend(
    handles=[mpatches.Patch(facecolor=COL_SPE1,    alpha=_ALPHA, label='spe-1  (all)'),
             mpatches.Patch(facecolor=COL_SPE1_PC,  alpha=_ALPHA, label='spe-1  PC'),
             mpatches.Patch(facecolor=COL_SPE1_IN,  alpha=_ALPHA, label='spe-1  IN'),
             mpatches.Patch(facecolor=COL_PVC6,     alpha=_ALPHA, label='pvc-6')],
    loc='center', frameon=False, ncol=2,
    prop={'size': _FS_TK, 'weight': 'bold'})
plt.tight_layout()
plt.show()